# OCR Bilans Fiscaux Algériens — V2 — H100 + Qwen3.6-VL-27B (bf16)

> **Documentation data scientists** : chaque cellule de code est précédée d'une fiche (Objectif / Entrées / Sorties / Règles / Limites) et commence par un en-tête commenté de traçabilité.

## Architecture du pipeline
1. Rendu PDF → images (zoom 3.0) + **deskew** automatique (±12°, scans de guichet inclinés)
2. Classification par page : `ACTIF` / `PASSIF` / `TCR` / `DECL` / `AUTRE`
3. Extraction JSON à **clés canoniques** (schéma unique partagé prompt ↔ normalisation ↔ Excel)
4. Contrôles inter-pages : même client (raison sociale fuzzy), même NIF, même année ; page étrangère → exclue + anomalie
5. Fusion TCR (2 pages) ; JSON immédiat (checkpoint anti-crash) ; Excel **2 lignes par bilan** (N et N-1)

## Journal des versions
| Version | Date | Changement |
|---|---|---|
| V1 | 2026-08-14 | Première version (colonnes `_n`/`_n1` sur 1 ligne) |
| V2 | 2026-08-14 | 2 lignes/an (`exercice`, `annee_exercice`, `annee_depot` via page DECL) ; deskew ; règle anti-cachets ; contrôles cohérence ; documentation DS + data dictionary |

## Conventions
- **JSON = source de vérité** ; Excel = vue de lecture reconstruite depuis les JSON
- Clés snake_case stables ; montants en `float` (dinars) ; **parenthèses = négatif**
- `annee_exercice` N = année de clôture (« Exercice clos le 31/12/2025 » → 2025) ; N-1 = N−1
- `annee_depot` = « Année de souscription » de la déclaration IBS (null si page absente)
- Anomalies en rouge dans l'Excel (client différent, année différente, NIF différent, doublons, manquants)

## Cellule 1 — Dépendances
**Objectif** : installer le runtime (environnement offline : commenter et passer par le miroir PyPI interne).
**Contrainte** : `transformers >= 4.57` requis (architecture Qwen3-VL + intégration FP8 `FineGrainedFP8Config`).
**Limite** : en offline strict, vérifier la présence des wheels sur le miroir avant le run.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 1 — DÉPENDANCES | Projet : BILANS_V2 | 2026-08-14
# Rôle : runtime OCR (transformers, pymupdf, openpyxl)
# ════════════════════════════════════════════════════════════
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow openpyxl psutil pandas
print('✅ OK')

## Cellule 2 — Imports
**Objectif** : charger les librairies.
| Lib | Rôle |
|---|---|
| `fitz` (PyMuPDF) | rendu vectoriel des pages PDF |
| `torch` / `transformers` | inférence VLM Qwen3.6-VL |
| `openpyxl` | écriture Excel multi-onglets stylée |
| `difflib` | similarité raison sociale (contrôle même client) |

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 2 — IMPORTS | BILANS_V2 | 2026-08-14
# ════════════════════════════════════════════════════════════
import time, json, re, gc, difflib
import numpy as np
from pathlib import Path
from datetime import datetime
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

## Cellule 3 — Config
**Objectif** : chemins E/S + hyperparamètres.
**Choix documentés** :
- `PDF_ZOOM=3.0` : résolution suffisante pour les petites cellules de chiffres
- `MAX_PIXELS=2000*32*32` (~2 MP) : facteur 32 = patch 16 × merge 2 du ViT Qwen3-VL ; tables denses → plus de pixels que le projet transferts
- `GPU_BATCH_SIZE=8` : pages par appel GPU (H100 80GB, modèle bf16 ~54 GB)
- `ANNEE_ATTENDUE` : si fixé (ex '2025'), tout dossier d'une autre année → anomalie
**Sorties** : `pdfs` (liste des PDF à traiter).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 3 — CONFIG | BILANS_V2 | 2026-08-14
# Entrées : chemins ModelHub / data | Sorties : pdfs, EXCEL_PATH
# ════════════════════════════════════════════════════════════
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'   # ← AJUSTER

DEVICE         = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS = 3000
IMAGE_MAX_SIZE = 2024
MIN_PIXELS     = 4 * 32 * 32
MAX_PIXELS     = 2000 * 32 * 32     # tables denses
PDF_ZOOM       = 3.0
BLANK_THRESHOLD= 0.95
GPU_BATCH_SIZE = 8
ANNEE_ATTENDUE = None               # ex '2025' → anomalie si dossier d'une autre année

INPUT_DIR  = Path('/mnt/data/bilans_in')
OUTPUT_DIR = Path('/mnt/data/bilans_out')
JSON_DIR   = OUTPUT_DIR / 'json_bilans'
LOG_PATH   = OUTPUT_DIR / 'pipeline_bilans.log'
EXCEL_PATH = OUTPUT_DIR / f'bilans_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device : {DEVICE} | Dossiers : {len(pdfs)} | Excel : {EXCEL_PATH}')

## Cellule 4 — Chargement modèle
**Objectif** : charger Qwen3.6-27B-FP8 **déquantisé en bf16**.
**Décision technique (V1→V2)** : le checkpoint FP8 block-wise (128×128) exige le kernel `kernels-community/finegrained-fp8`, indisponible en offline → `FineGrainedFP8Config(dequantize=True)` convertit les poids en bf16 au load. Coût : ~54 GB VRAM (tenable sur H100 80GB) ; bénéfice : zéro dépendance externe, calcul exact.
**Réglages** : `padding_side='left'` (génération batch) ; `enable_thinking=False` (extraction déterministe).
**Sorties** : `processor`, `model`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 4 — CHARGEMENT MODÈLE | BILANS_V2 | 2026-08-14
# FP8 → bf16 (dequantize) | padding gauche | thinking off
# ════════════════════════════════════════════════════════════
t0 = time.time()
processor = AutoProcessor.from_pretrained(MODEL_PATH, trust_remote_code=True,
                                          min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side = 'left'

try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=torch.bfloat16, device_map='auto',
    trust_remote_code=True, low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s | VRAM libre : {torch.cuda.mem_get_info()[0]/1e9:.1f} GB')

## Cellule 5 — Utilitaires PDF & inférence
**Objectif** : rendu, redressement, détection pages vides, appel modèle.
**`deskew`** : méthode du profil de projection (variance des sommes de lignes) sur thumbnail 500 px, recherche −12°→+12° pas 2° puis raffinement ±1° pas 0.5° ; rotation appliquée seulement si |angle| ≥ 1° (ignore les micro-inclinaisons).
**`ask_batch`** : N images / 1 appel GPU, padding gauche ; décodage **greedy** (`do_sample=False`) → reproductible ; `clean_up_tokenization_spaces=False` (le cleanup est destructif pour les tokenizers BPE).
**Sorties** : `{text, tokens_in, tokens_out, elapsed}` par page (comptabilité tokens pour le coût).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 5 — UTILITAIRES | BILANS_V2 | 2026-08-14
# deskew (profil de projection) + inférence batch reproductible
# ════════════════════════════════════════════════════════════
def resize(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side: return img
    r = max_side / max(w, h)
    return img.resize((int(w*r), int(h*r)), Image.LANCZOS)

def estimate_skew(img):
    """Angle d'inclinaison (degrés) par variance du profil de projection horizontale."""
    small = img.convert('L').copy(); small.thumbnail((500, 500))
    def score(a):
        r = np.array(small.rotate(a, expand=True, fillcolor=255)) < 128
        proj = r.sum(axis=1)
        return float((proj ** 2).sum())
    best = max(range(-12, 13, 2), key=score)
    best = max([best-1, best-0.5, best, best+0.5, best+1], key=score)
    return best if abs(best) >= 1 else 0.0

def deskew(img):
    a = estimate_skew(img)
    if a: img = img.rotate(a, expand=True, fillcolor=(255,255,255), resample=Image.BICUBIC)
    return img

def is_blank(image, threshold=BLANK_THRESHOLD) -> bool:
    arr = np.array(image.convert('L'))
    return (arr > 240).sum() / arr.size >= threshold

def pdf_to_pages(path: Path, zoom=PDF_ZOOM) -> list:
    doc = fitz.open(path); matrix = fitz.Matrix(zoom, zoom); pages = []
    for i in range(len(doc)):
        pix = doc.load_page(i).get_pixmap(matrix=matrix, alpha=False)
        pages.append({'index': i, 'image': resize(deskew(Image.frombytes('RGB', [pix.width, pix.height], pix.samples)))})
    doc.close()
    return pages

def parse_json(text: str) -> dict:
    """Extrait le premier bloc {...} de la sortie modèle (tolère un texte parasite)."""
    try:
        m = re.search(r'\{.*\}', text, re.S)
        return json.loads(m.group()) if m else {}
    except Exception:
        return {}

def apply_template(messages) -> str:
    try:
        return processor.apply_chat_template(messages, tokenize=False,
            add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def _decode(out_i, in_len):
    return processor.decode(out_i[in_len:], skip_special_tokens=True,
                            clean_up_tokenization_spaces=False)

def ask_single(prompt, image) -> dict:
    msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs = processor(text=[apply_template(msgs)], images=[image], return_tensors='pt').to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text': _decode(out[0], inputs['input_ids'].shape[1]),
            'tokens_in': int(inputs['input_ids'].shape[1]),
            'tokens_out': int(out[0].shape[0] - inputs['input_ids'].shape[1]),
            'elapsed': round(time.time()-t0, 2)}

def ask_batch(prompt, images) -> list:
    """N images en 1 appel GPU (même prompt). Retourne liste de dicts stats."""
    if not images: return []
    if len(images) == 1: return [ask_single(prompt, images[0])]
    msgs = [[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs = processor(text=[apply_template(m) for m in msgs], images=images,
                       return_tensors='pt', padding=True).to(DEVICE)
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             repetition_penalty=1.0, pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    el = time.time() - t0
    in_len = inputs['input_ids'].shape[1]
    attn = inputs.get('attention_mask')
    return [{'text': _decode(out[i], in_len),
             'tokens_in': int(attn[i].sum().item()) if attn is not None else in_len,
             'tokens_out': int(out[i].shape[0] - in_len), 'elapsed': round(el/len(images), 2)}
            for i in range(len(images))]

print('✅ Utilitaires OK (deskew inclus)')

## Cellule 6 — Schémas canoniques + prompt généré
**Objectif** : définir le **contrat de données** unique (clés ↔ libellés imprimés) et générer le prompt depuis ce schéma → garantie que prompt, normalisation et Excel partagent exactement les mêmes clés.
**Tables** : `ACTIF` (brut/amort/n/n1), `PASSIF` (n/n1), `TCR` (n/n1, ~48 rubriques dans l'ordre de l'imprimé Série G).
**Types de page** : ACTIF, PASSIF, TCR, DECL (déclaration IBS → `annee_souscription`), AUTRE (annexes 2/-13/, TAP… ignorées).
**Règles scan injectées dans le prompt** : inclinaison tolérée ; cachets/tampons ignorés (lire la valeur imprimée dessous) ; parenthèses = négatif ; colonnes dupliquées lues une seule fois.
**Limite connue** : libellés OCRisés dans le prompt (sans accents) pour rapprochement robuste.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 6 — SCHÉMAS + PROMPT | BILANS_V2 | 2026-08-14
# SCHEMAS = contrat de données ; PROMPT_BILAN généré depuis SCHEMAS
# ════════════════════════════════════════════════════════════
SCHEMAS = {
 'ACTIF': {'cols': ['brut','amort','n','n1'], 'postes': {
   'ecarts_acquisition_goodwill': 'Ecart d acquisition goodwill',
   'immobilisations_incorporelles': 'Immobilisations incorporelles',
   'terrains': 'Terrains',
   'batiments': 'Batiments',
   'autres_immobilisations_corporelles': 'Autres Immobilisations corporelles',
   'immobilisations_en_concession': 'Immobilisations en concession',
   'immobilisations_en_cours': 'Immobilisations en cours',
   'titres_mis_en_equivalence': 'Titres mis en equivalence',
   'autres_participations_creances': 'Autres participations et creances rattachees',
   'autres_titres_immobilises': 'Autres titres immobilises',
   'prets_actifs_financiers_non_courants': 'Prets et autres actifs financiers non courants',
   'impots_differes_actif': 'Impots Differes Actif',
   'total_actif_non_courant': 'TOTAL ACTIF NON COURANT',
   'stocks_encours': 'Stocks et encours',
   'clients': 'Clients',
   'autres_debiteurs': 'Autres debiteurs',
   'impots_assimiles_actif': 'Impots & Assimiles',
   'autres_creances_assimiles': 'Autres Creances & Emplois assimiles',
   'placements_financiers_courants': 'Placements et autres actifs financiers courants',
   'tresorerie_actif': 'Tresorerie',
   'total_actif_courant': 'TOTAL ACTIF COURANT',
   'total_general_actif': 'TOTAL GENERAL ACTIF'}},
 'PASSIF': {'cols': ['n','n1'], 'postes': {
   'capital_emis': 'Capital emis (ou compte de l exploitant)',
   'capital_non_appele': 'Capital non appele',
   'primes_reserves': 'Primes et reserves (Reserves consolidees)',
   'ecart_reevaluation': 'Ecart de reevaluation',
   'ecart_equivalence': 'Ecart d equivalence (1)',
   'resultat_net_passif': 'Resultat net (Resultat net part du groupe) (1)',
   'report_a_nouveau': 'Autres capitaux propres - Report a nouveau',
   'part_societe_consolidante': 'Part de la societe consolidante (1)',
   'part_minoritaires': 'Part des minoritaires (1)',
   'total_capitaux_propres': 'TOTAL I',
   'emprunts_dettes_financieres': 'Emprunts et dettes financieres',
   'impots_differes_provisionnes': 'Impots differes et provisionnes',
   'autres_dettes_non_courantes': 'Autres dettes non courantes',
   'provisions_produits_avance': 'Provisions et produits comptabilises d avance',
   'total_passifs_non_courants': 'TOTAL PASSIFS NON COURANTS II',
   'fournisseurs_rattaches': 'Fournisseurs et comptes rattaches',
   'impots_passif': 'Impots',
   'autres_dettes': 'Autres dettes',
   'tresorerie_passif': 'Tresorerie Passif',
   'total_passifs_courants': 'TOTAL PASSIFS COURANTS (II ou III)',
   'total_general_passif': 'TOTAL GENERAL PASSIF'}},
 'TCR': {'cols': ['n','n1'], 'postes': {
   'ventes_marchandises': 'Ventes de Marchandises',
   'produits_fabriques': 'Produits Fabriques',
   'prestations_services': 'Prestations de Services',
   'ventes_travaux': 'Ventes de Travaux',
   'produits_annexes': 'Produits Annexes',
   'rabais_remises_ristournes_accordes': 'Rabais, remises, ristournes accordes',
   'chiffre_affaires_net': 'Chiffre d affaires net des Rabais, remises, ristournes',
   'production_stockee_destockee': 'Production Stockee ou destockee',
   'production_immobilisee': 'Production immobilisee',
   'subvention_exploitation': 'Subvention d exploitation',
   'production_exercice': 'I-Production de l exercice',
   'achats_marchandises_vendues': 'Achats de Marchandises vendues',
   'matieres_premieres': 'Matieres premieres',
   'autres_approvisionnements': 'Autres Approvisionnements',
   'variation_stocks': 'Variation des Stocks',
   'achats_etudes_prestations': 'Achats d Etudes et de Prestations de services',
   'autres_consommations': 'Autres consommations',
   'sous_traitance_generale': 'Sous-traitance generale',
   'locations': 'Locations',
   'entretien_reparations': 'Entretien, reparations et maintenance',
   'primes_assurances': 'Primes d assurances',
   'personnel_exterieur': 'Personnel exterieur a l entreprise',
   'remuneration_intermediaires': 'Remuneration d intermediaires et honoraires',
   'publicite': 'Publicite',
   'deplacements_missions': 'Deplacements, missions et receptions',
   'autres_services': 'Autres services',
   'consommations_exercice': 'II-Consommations de l exercice',
   'valeur_ajoutee_exploitation': 'III-Valeur ajoutee d exploitation (I-II)',
   'charges_personnel': 'Charges de personnel',
   'impots_taxes_assimiles': 'Impots et taxes et versements assimiles',
   'excedent_brut_exploitation': 'IV-Excedent brut d exploitation',
   'autres_produits_operationnels': 'Autres produits operationnels',
   'autres_charges_operationnelles': 'Autres charges operationnelles',
   'dotations_amortissements': 'Dotations aux amortissements',
   'provisions': 'Provisions',
   'pertes_valeur': 'Perte de Valeur',
   'reprises_pertes_valeur_provisions': 'Reprise sur pertes de valeur et provisions',
   'resultat_operationnel': 'V-Resultat operationnel',
   'produits_financiers': 'Produits financiers',
   'charges_financieres': 'Charges financieres',
   'resultat_financier': 'VI-Resultat Financier',
   'resultat_ordinaire': 'VII-Resultat ordinaire (V+VI)',
   'elements_extraordinaires_produits': 'Elements extraordinaires (Produits)',
   'elements_extraordinaires_charges': 'Elements extraordinaires (Charges)',
   'resultat_extraordinaire': 'VIII-Resultat extraordinaire',
   'impots_exigibles_resultats': 'Impots exigibles sur resultats',
   'impots_differes_resultats': 'Impots differes (variations) sur resultats',
   'resultat_net_exercice': 'RESULTAT NET DE L EXERCICE'}},
}

# ── Prompt généré depuis les schémas (clés prompt == clés normalisation == clés Excel) ──
L = []
L.append('Lis cette page d\'un dossier fiscal algerien (imprime Serie G).')
L.append('')
L.append('ETAPE 1 — Identifie le tableau principal :')
L.append('- ACTIF  : titre "BILAN (ACTIF)"')
L.append('- PASSIF : titre "BILAN (PASSIF)"')
L.append('- TCR    : titre "COMPTE DE RESULTAT" (parfois mal imprime "COMPTE DE REESULTAT")')
L.append('- DECL   : page "DECLARATION DE L\'IMPOT SUR LES BENEFICES DES SOCIETES" (entete avec "Annee de souscription")')
L.append('- AUTRE  : toute autre page (annexes, tableaux 2/ a 13/, TAP, page vide)')
L.append('')
L.append('ETAPE 2 — Extrais les montants du tableau identifie en JSON :')
L.append('{"type": ..., "entreprise": ..., "exercice": ..., "nif": ..., "annee_souscription": ..., "postes": {cle: {col: montant}}}')
L.append('- entreprise : raison sociale en haut de page | exercice : date de cloture (ex 31/12/2025) | nif : NIF si visible sinon null')
L.append('- annee_souscription : uniquement sur page DECL (ex 2026), sinon null')
L.append('- Colonnes : ACTIF → brut (Montants bruts), amort (Amortissements/Provisions), n (Net N), n1 (Net N-1) | PASSIF → n, n1 | TCR → n, n1')
L.append('- TCR : n = montant de l exercice N (colonne DEBIT ou CREDIT selon le sens du poste), n1 = idem pour N-1')
L.append('- Montant entre parentheses = negatif : (1 553 799) → -1553799')
L.append('- Montants en NOMBRES JSON sans espaces ni separateurs ; case vide ou illisible : null')
L.append('- Si des colonnes sont dupliquees ou decalees (scans), lis chaque valeur une seule fois dans la bonne colonne')
L.append('- Utilise EXACTEMENT les cles ci-dessous. Si AUTRE : {"type": "AUTRE"}')
for table, spec in SCHEMAS.items():
    L.append(f'--- Si {table} ---')
    for key, label in spec['postes'].items():
        L.append(f'{key} : ligne "{label}"')
L.append('')
L.append('PRECISIONS SCAN :')
L.append('- Les pages peuvent etre inclinees de quelques degres : lis normalement malgre l\'inclinaison.')
L.append('- Des cachets, tampons, signatures ou griffures peuvent recouvrir du texte ou des montants : ignore-les et lis la valeur IMPRIMEE en dessous.')
L.append('- entreprise, nif et exercice doivent etre lus en haut de CHAQUE page (controle de coherence).')
L.append('')
L.append('REGLES : JSON valide uniquement, sans texte avant/apres, pas de markdown, pas de backticks, aucun champ invente.')
PROMPT_BILAN = '\n'.join(L)
print(f'✅ Schémas + prompt OK ({len(PROMPT_BILAN)} caractères)')

In [ ]:
# ═══ CELLULE 6bis — PROMPT CLASSIF + SORTIE ALLÉGÉE ═══
PROMPT_CLASSIF = ('Page dun dossier fiscal algerien Serie G. Reponds UN seul mot : '
                  'ACTIF si titre BILAN (ACTIF) ; PASSIF si BILAN (PASSIF) ; '
                  'TCR si COMPTE DE RESULTAT ; DECL si page DECLARATION (Annee de souscription) ; '
                  'AUTRE sinon.')

PROMPT_BILAN += ("\n- IMPORTANT : dans postes, retourne UNIQUEMENT les cles avec une valeur "
                 "non nulle (les cles absentes seront mises a null automatiquement).")
print('✅ Two-pass prêt')

## Cellule 7 — Normalisation
**Objectif** : convertir les sorties modèle en valeurs typées stables.
**Règles `norm_montant`** : nombre JSON accepté tel quel ; chaîne → suppression séparateurs milliers ; virgule décimale → point ; **parenthèses ou signe − → négatif**.
**`normalise_table`** : boucle générique sur `SCHEMAS[table]` → clés de sortie `{cle}_{colonne}` (ex `terrains_n`, `terrains_n1`).
**Traçabilité** : toute valeur illisible → `null` (jamais de valeur inventée).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 7 — NORMALISATION | BILANS_V2 | 2026-08-14
# Sorties modèle → float/str typés ; clés {cle}_{col}
# ════════════════════════════════════════════════════════════
def norm_montant(v):
    """Chaîne/nombre → float. Parenthèses ou '-' = négatif ; séparateurs milliers gérés."""
    if v is None: return None
    if isinstance(v, (int, float)): return float(v)
    s = str(v).strip()
    neg = (s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s = re.sub(r'[^\d.,]', '', s)
    if not s: return None
    if s.count(',') == 1 and '.' not in s: s = s.replace(',', '.')
    elif ',' in s: s = s.replace(',', '')
    elif s.count('.') > 1: s = s.replace('.', '')
    try: return -float(s) if neg else float(s)
    except Exception: return None

def norm_str(v):
    if v is None: return None
    s = re.sub(r'\s+', ' ', str(v).strip())
    return s if s and s.lower() not in ('null','none','n/a') else None

def norm_upper(v):
    s = norm_str(v)
    return s.upper() if s else None

def norm_nif(v):
    s = norm_str(v)
    return re.sub(r'[^0-9]', '', s) if s else None

def norm_annee4(v):
    s = norm_str(v) or ''
    m = re.findall(r'20\d{2}', s)
    return m[-1] if m else None

def normalise_table(table, data):
    """data (sortie modèle) → dict plat {cle}_{col}: float|None, selon SCHEMAS[table]."""
    spec = SCHEMAS[table]
    raw = data.get('postes') or {}
    out = {}
    for key in spec['postes']:
        vals = raw.get(key)
        vals = vals if isinstance(vals, dict) else {}
        for col in spec['cols']:
            out[f'{key}_{col}'] = norm_montant(vals.get(col))
    return out

print('✅ Normalisation OK')

## Cellule 8 — Cohérence inter-pages (anti-fraude / anti-mélange)
**Objectif** : vérifier que toutes les pages d'un PDF appartiennent au même dossier.
**Méthode** : vote majoritaire sur (raison sociale normalisée, année 20XX, NIF) ; similarité raison sociale = égalité / inclusion / `SequenceMatcher ratio > 0.80`.
**Décisions** :
- page d'un **client différent** → exclue des tableaux + anomalie `PAGE x: CLIENT DIFFERENT`
- **année différente** ou ≠ `ANNEE_ATTENDUE` → anomalie (page conservée, à arbitrer humainement)
- **NIF différent** → anomalie (contrôle secondaire, NIF souvent OCRisé)
**Sorties** : `(ref_entreprise, ref_nif, ref_annee, anomalies, index_exclus)`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 8 — CONTRÔLE COHÉRENCE | BILANS_V2 | 2026-08-14
# Même client / même NIF / même année sur toutes les pages
# ════════════════════════════════════════════════════════════
def norm_identite(s):
    s = norm_upper(s)
    return re.sub(r'[^A-Z0-9]', '', s) if s else None

def meme_entreprise(a, b):
    if not a or not b: return True
    if a == b or a in b or b in a: return True
    return difflib.SequenceMatcher(None, a, b).ratio() > 0.80

def controle_coherence(parsed, annee_attendue=None):
    """parsed : [{'index','type','meta','data'}] → (ref_ent, ref_nif, ref_annee, anomalies, exclus)."""
    anomalies, excluded = [], set()
    idents = [(p['index'], norm_identite(p['meta'].get('entreprise'))) for p in parsed]
    vals = [e for _, e in idents if e]
    ref_ent = max(set(vals), key=lambda e: sum(1 for x in vals if meme_entreprise(e, x))) if vals else None
    for idx, e in idents:
        if ref_ent and e and not meme_entreprise(ref_ent, e):
            anomalies.append(f'PAGE {idx+1}: CLIENT DIFFERENT')
            excluded.add(idx)
    annees = [(p['index'], norm_annee4(p['meta'].get('exercice'))) for p in parsed]
    yy = [a for _, a in annees if a]
    ref_annee = max(set(yy), key=yy.count) if yy else None
    for idx, a in annees:
        if ref_annee and a and a != ref_annee:
            anomalies.append(f'PAGE {idx+1}: ANNEE {a} ≠ {ref_annee}')
    if annee_attendue and ref_annee and ref_annee != str(annee_attendue):
        anomalies.append(f'ANNEE DOSSIER {ref_annee} ≠ ATTENDUE {annee_attendue}')
    nifs = [(p['index'], norm_nif(p['meta'].get('nif'))) for p in parsed]
    nn = [n for _, n in nifs if n and len(n) >= 12]
    ref_nif = max(set(nn), key=nn.count) if nn else None
    for idx, n in nifs:
        if ref_nif and n and len(n) >= 12 and n != ref_nif:
            anomalies.append(f'PAGE {idx+1}: NIF DIFFERENT')
    return ref_ent, ref_nif, ref_annee, anomalies, excluded

print('✅ Contrôle cohérence OK')

## Cellule 9 — Export Excel (2 lignes par bilan)
**Objectif** : vue analyste — **1 ligne = 1 exercice** (N et N-1), colonnes `exercice`, `annee_exercice`, `annee_depot`.
**Règles d'affichage** : valeurs = `_{n}` sur la ligne N, `_{n1}` sur la ligne N-1 ; colonnes `brut`/`amort` remplies **uniquement sur la ligne N** (inexistantes en N-1 dans l'imprimé) ; séparateur de milliers ; anomalies en rouge gras ; volet figé après les métadonnées.
**Entrées** : `rows` (JSON chargés) | **Sortie** : fichier `.xlsx`.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 9 — EXPORT EXCEL | BILANS_V2 | 2026-08-14
# 2 lignes/bilan (N, N-1) ; colonnes exercice/annee_exercice/annee_depot
# ════════════════════════════════════════════════════════════
COLORS = {'META':'FFD6E4F0', 'ACTIF':'FFE2EFDA', 'PASSIF':'FFDAE3F3', 'TCR':'FFFCE4D6'}
HDRS   = {'META':'FF1F4E79', 'ACTIF':'FF375623', 'PASSIF':'FF203864', 'TCR':'FF833C00'}
TOT_ACT = ['total_actif_non_courant','total_actif_courant','total_general_actif']

def build_cols():
    cols = [('META', k, lab) for k, lab in [
        ('fichier','Fichier'), ('entreprise','Entreprise'), ('nif','NIF'),
        ('exercice','Exercice'), ('annee_exercice','Année exercice'),
        ('annee_depot','Année dépôt'), ('date_traitement','Date traitement'),
        ('temps_total_s','Temps (s)'), ('tokens_in','Tokens IN'),
        ('tokens_out','Tokens OUT'), ('tokens_total','Tokens total'),
        ('pages_trouvees','Tableaux trouvés'), ('anomalies','Anomalies')]]
    for key in SCHEMAS['ACTIF']['postes']: cols.append(('ACTIF', key, key))
    for key in TOT_ACT:
        cols.append(('ACTIF', key+'_brut', key+' [brut]'))
        cols.append(('ACTIF', key+'_amort', key+' [amort]'))
    for key in SCHEMAS['PASSIF']['postes']: cols.append(('PASSIF', key, key))
    for key in SCHEMAS['TCR']['postes']: cols.append(('TCR', key, key))
    return cols

def val_for(d, groupe, key, suf):
    """Valeur Excel : suf='n' (ligne N) ou 'n1' (ligne N-1) ; brut/amort seulement en N."""
    t = d.get(groupe) or {}
    if groupe == 'ACTIF' and (key.endswith('_brut') or key.endswith('_amort')):
        return t.get(key) if suf == 'n' else None
    return t.get(f'{key}_{suf}')

def create_excel(path, rows):
    from collections import defaultdict
    all_cols = build_cols()
    wb = Workbook(); ws = wb.active; ws.title = 'Bilans'
    grp = defaultdict(list); idx = 1
    for g, _, _ in all_cols: grp[g].append(idx); idx += 1
    for g, cs in grp.items():
        s, e = cs[0], cs[-1]
        if s < e: ws.merge_cells(start_row=1, start_column=s, end_row=1, end_column=e)
        c = ws.cell(row=1, column=s); c.value = g
        c.font = Font(bold=True, color='FFFFFFFF', name='Arial', size=11)
        c.fill = PatternFill('solid', start_color=HDRS[g])
        c.alignment = Alignment(horizontal='center', vertical='center')
    for i, (g, _, label) in enumerate(all_cols, start=1):
        c = ws.cell(row=2, column=i); c.value = label
        c.font = Font(bold=True, name='Arial', size=8)
        c.fill = PatternFill('solid', start_color=COLORS[g])
        c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        ws.column_dimensions[get_column_letter(i)].width = 16
    ws.row_dimensions[2].height = 30
    ws.freeze_panes = ws.cell(row=3, column=7)
    rn = 3
    for d in rows:
        annee_n = norm_str(d.get('exercice'))
        annee_n1 = str(int(annee_n)-1) if annee_n and annee_n.isdigit() else None
        for exer, annee, suf in [('N', annee_n, 'n'), ('N-1', annee_n1, 'n1')]:
            for ci, (g, key, _) in enumerate(all_cols, start=1):
                if g == 'META':
                    if key == 'exercice': val = exer
                    elif key == 'annee_exercice': val = annee
                    else: val = d.get(key)
                else:
                    val = val_for(d, g, key, suf)
                c = ws.cell(row=rn, column=ci); c.value = val
                c.font = Font(name='Arial', size=9)
                c.fill = PatternFill('solid', start_color=COLORS[g])
                if isinstance(val, float): c.number_format = '#,##0.00'
                if g == 'META' and key == 'anomalies' and val:
                    c.font = Font(name='Arial', size=9, bold=True, color='FFCC0000')
            rn += 1
    wb.save(path)
    print(f'✅ Excel : {path} | {len(rows)} bilans → {rn-3} lignes | {len(all_cols)} colonnes')

print('✅ Export Excel OK (2 lignes/bilan)')

## Cellule 10 — Log
**Objectif** : journal horodaté miroir (stdout + `pipeline_bilans.log`) pour audit et suivi live (`tail -f`).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 10 — LOG | BILANS_V2 | 2026-08-14
# ════════════════════════════════════════════════════════════
def log(msg: str):
    ligne = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH, 'a', encoding='utf-8') as f: f.write(ligne + '\n')
print('✅ Log OK')

## Cellule 11 — Pipeline (1 PDF = 1 bilan)
**Flux** : rendu+deskew → inférence batch 8 pages → contrôle cohérence → fusion TCR (2 pages, 1er non-null gagne) → normalisation → **JSON immédiat** (checkpoint anti-crash) → log live avec ETA → Excel final.
**Métadonnées tracées** : entreprise, NIF, `exercice` (année N), `annee_depot` (page DECL), date/temps de traitement, tokens IN/OUT, tableaux trouvés, anomalies.
**Reprise** : les JSON existants servent de checkpoint (`a_traiter` = PDFs sans JSON).

In [ ]:
# ═══ CELLULE 11 v3 — PIPELINE TWO-PASS ═══
deja = {f.stem for f in JSON_DIR.glob('*.json')}
a_traiter = [p for p in pdfs if p.stem not in deja]
log(f'À traiter : {len(a_traiter)} | déjà traités : {len(deja)}')

t_total = time.time(); n_ok = n_err = 0
TYPES = {'ACTIF', 'PASSIF', 'TCR'}

def parse_type(text):
    t = (text or '').upper()
    for k in ('ACTIF', 'PASSIF', 'TCR', 'DECL'):
        if k in t: return k
    return 'AUTRE'

for num, pdf_path in enumerate(a_traiter, start=1):
    t0 = time.time()
    try:
        pages   = pdf_to_pages(pdf_path)
        actives = [p for p in pages if not is_blank(p['image'])]
        tok_in = tok_out = 0

        # ── PASS 1 : classification low-cost (mini 600px, mini prompt) ──
        mini = [resize(p['image'], 600) for p in actives]
        reps1 = []
        for bs in range(0, len(mini), 16):
            reps1 += ask_batch(PROMPT_CLASSIF, mini[bs:bs+16])
        tok_in  += sum(r['tokens_in']  for r in reps1)
        tok_out += sum(r['tokens_out'] for r in reps1)
        utiles = [(p, parse_type(r['text'])) for p, r in zip(actives, reps1)
                  if parse_type(r['text']) != 'AUTRE']

        # ── PASS 2 : extraction full-res sur les pages utiles seulement ──
        parsed, annee_depot = [], None
        for bs in range(0, len(utiles), GPU_BATCH_SIZE):
            batch = utiles[bs:bs+GPU_BATCH_SIZE]
            reps  = ask_batch(PROMPT_BILAN, [p['image'] for p, _ in batch])
            for (page, tpage), rep in zip(batch, reps):
                tok_in += rep['tokens_in']; tok_out += rep['tokens_out']
                data = parse_json(rep['text'])
                t = data.get('type', tpage)
                if t not in TYPES and t != 'DECL': continue
                meta = {k: data.get(k) for k in ('entreprise', 'nif')}
                meta['exercice'] = None if t == 'DECL' else data.get('exercice')
                if t == 'DECL':
                    dep = norm_annee4(data.get('annee_souscription'))
                    if dep and not annee_depot: annee_depot = dep
                parsed.append({'index': page['index'], 'type': t, 'meta': meta, 'data': data})
            gc.collect(); torch.cuda.empty_cache()

        # ── Cohérence + fusion TCR (identique v2) ──
        ref_ent, ref_nif, ref_annee, anomalies, excluded = controle_coherence(parsed, ANNEE_ATTENDUE)
        ent_raw = next((norm_str(p['meta'].get('entreprise')) for p in parsed
                        if p['index'] not in excluded
                        and meme_entreprise(ref_ent or '', norm_identite(p['meta'].get('entreprise')))), None)
        tables, tcr_pages, doublons = {}, [], []
        for p in parsed:
            if p['index'] in excluded or p['type'] == 'DECL': continue
            t = p['type']
            if t == 'TCR': tcr_pages.append(p['data'])
            else:
                if t in tables: doublons.append(t); continue
                tables[t] = normalise_table(t, p['data'])
        tcr = {}
        for d in tcr_pages:
            for k, v in normalise_table('TCR', d).items():
                if tcr.get(k) is None and v is not None: tcr[k] = v
        if tcr: tables['TCR'] = tcr
        if doublons: anomalies.append('DOUBLON: ' + ', '.join(doublons))

        dt = round(time.time() - t0, 2)
        manquants = sorted(TYPES - set(tables.keys()))
        result = {
            'fichier': pdf_path.name,
            'entreprise': ent_raw, 'nif': ref_nif,
            'exercice': ref_annee, 'annee_depot': annee_depot,
            'date_traitement': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'temps_total_s': dt, 'tokens_in': tok_in, 'tokens_out': tok_out,
            'tokens_total': tok_in + tok_out,
            'pages_trouvees': ', '.join(sorted(tables.keys())),
            'anomalies': ' | '.join(anomalies) if anomalies else None,
            **{t: tables.get(t, {}) for t in TYPES},
        }
        with open(JSON_DIR / f'{pdf_path.stem}.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, ensure_ascii=False, indent=2, default=str)
        n_ok += 1
        eta = (time.time() - t_total) / num * (len(a_traiter) - num)
        msg = f'[{num:>4}/{len(a_traiter)}] ✅ {pdf_path.name} | {dt:.1f}s | tok={tok_in+tok_out} | {result["pages_trouvees"]}'
        if manquants:         msg += f' | ⚠️  manquants: {", ".join(manquants)}'
        if result['anomalies']: msg += f' | 🔴 {result["anomalies"]}'
        log(msg + f' | ETA {eta/3600:.1f}h')
    except Exception as e:
        n_err += 1
        log(f'[{num:>4}/{len(a_traiter)}] ❌ {pdf_path.name} — {e}')
        continue

log('Génération Excel...')
rows = [json.load(open(jf, encoding='utf-8')) for jf in sorted(JSON_DIR.glob('*.json'))]
create_excel(EXCEL_PATH, rows)
log(f'✅ Terminé en {time.time()-t_total:.1f}s | OK {n_ok} | Erreurs {n_err}')

## Cellule 12 — Contrôles comptables
**Objectif** : validation croisée par année : `Total actif = Total passif` (N et N-1) et `Résultat net TCR = Résultat net au passif` (N).
**Usage** : aucune ligne affichée = cohérent ; sinon → dossier à revoir (OCR ou document source).

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 12 — CONTRÔLES COMPTABLES | BILANS_V2 | 2026-08-14
# Actif=Passif (N & N-1) ; RN TCR = RN passif (N)
# ════════════════════════════════════════════════════════════
for d in rows:
    a, p, t = d.get('ACTIF') or {}, d.get('PASSIF') or {}, d.get('TCR') or {}
    for suf, lab in [('n', 'N'), ('n1', 'N-1')]:
        ta, tp = a.get(f'total_general_actif_{suf}'), p.get(f'total_general_passif_{suf}')
        if ta and tp and abs(ta - tp) > 1:
            print(f'❌ {d["fichier"]} [{lab}] : Actif {ta:,.0f} ≠ Passif {tp:,.0f}')
    rn_t, rn_p = t.get('resultat_net_exercice_n'), p.get('resultat_net_passif_n')
    if rn_t is not None and rn_p is not None and abs(rn_t - rn_p) > 1:
        print(f'❌ {d["fichier"]} : Résultat TCR {rn_t:,.0f} ≠ Passif {rn_p:,.0f}')
print('🔎 Contrôles terminés (aucune ligne = cohérent)')

## Cellule 13 — Data Dictionary (traçabilité DS)
**Objectif** : exporter le contrat de données (`SCHEMAS`) en CSV lisible : table / clé / libellé imprimé / colonnes → référence pour audits, onboarding DS et évolutions du schéma.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELLULE 13 — DATA DICTIONARY | BILANS_V2 | 2026-08-14
# Export CSV du schéma (clé ↔ libellé imprimé ↔ colonnes)
# ════════════════════════════════════════════════════════════
import csv
dd_path = OUTPUT_DIR / 'data_dictionary_bilans.csv'
with open(dd_path, 'w', newline='', encoding='utf-8') as f:
    w = csv.writer(f)
    w.writerow(['table', 'cle', 'libelle_imprime', 'colonnes'])
    for t, spec in SCHEMAS.items():
        for k, lab in spec['postes'].items():
            w.writerow([t, k, lab, '|'.join(spec['cols'])])
print(f'✅ Data dictionary : {dd_path} ({sum(len(s["postes"]) for s in SCHEMAS.values())} postes)')